In [1]:
import requests
import pandas as pd
from io import BytesIO
import datetime
import os

def get_bapanas_dataframe(tanggal, level_harga_id=3, province_id=15, filter_beras=False):
    """
    Mengambil data dari API BAPANAS dan langsung mengembalikan DataFrame
    tanpa menyimpan file Excel.

    Args:
        tanggal: datetime.date object atau string format 'YYYY-MM-DD'
        level_harga_id: int (3=konsumen, 1=produsen, dst)
        province_id: int (15=Jawa Timur, default)
        filter_beras: bool (True=hanya Beras Premium & Medium, False=semua data)

    Returns:
        tuple: (success: bool, dataframe: pd.DataFrame, message: str)

    Example:
        >>> import datetime
        >>> success, df, msg = get_bapanas_dataframe(datetime.date(2024, 1, 15), filter_beras=True)
        >>> if success:
        >>>     print(df.head())
    """
    # Konversi string ke datetime.date jika perlu
    if isinstance(tanggal, str):
        tanggal = datetime.datetime.strptime(tanggal, '%Y-%m-%d').date()

    # Format tanggal untuk API
    formatted = tanggal.strftime("%d/%m/%Y")
    period_param = f"{formatted}%20-%20{formatted}"

    # URL API BAPANAS
    url = f"https://api-panelhargav2.badanpangan.go.id/harga-pangan-table-province/export?province_id={province_id}&period_date={period_param}&level_harga_id={level_harga_id}"

    try:
        response = requests.get(url, timeout=30)

        if response.status_code == 200:
            # Baca Excel langsung dari memory tanpa menyimpan ke file
            df = pd.read_excel(BytesIO(response.content))

            if df.empty:
                return False, df, "Data kosong atau tidak tersedia untuk tanggal tersebut"

            # Filter hanya Beras Premium dan Beras Medium jika diminta
            if filter_beras:
                # Cari kolom identifikasi (No, Kota/Kabupaten, dll)
                kolom_identitas = []
                for col in df.columns:
                    col_lower = col.lower()
                    if any(keyword in col_lower for keyword in ['no', 'kota', 'kabupaten', 'wilayah', 'daerah', 'provinsi']):
                        kolom_identitas.append(col)

                # Cari kolom Beras Premium dan Beras Medium
                kolom_beras = []
                for col in df.columns:
                    if 'Beras Premium' in col or 'Beras Medium' in col:
                        kolom_beras.append(col)

                if not kolom_beras:
                    return False, pd.DataFrame(), "Kolom Beras Premium atau Beras Medium tidak ditemukan"

                # Pilih hanya kolom identitas + kolom beras
                kolom_terpilih = kolom_identitas + kolom_beras
                df = df[kolom_terpilih]

                if df.empty:
                    return False, df, "Data kosong setelah filter"

                return True, df, f"Berhasil mengambil {len(df)} baris data (hanya kolom {', '.join(kolom_beras)})"

            return True, df, f"Berhasil mengambil {len(df)} baris data"
        else:
            return False, pd.DataFrame(), f"HTTP Error {response.status_code}"

    except requests.exceptions.Timeout:
        return False, pd.DataFrame(), "Request timeout - API tidak merespons"
    except requests.exceptions.RequestException as e:
        return False, pd.DataFrame(), f"Request error: {str(e)}"
    except Exception as e:
        return False, pd.DataFrame(), f"Error: {str(e)}"


def get_bapanas_konsumen_produsen(tanggal, province_id=15, filter_beras=False):
    """
    Mengambil data konsumen DAN produsen sekaligus dari API BAPANAS.

    Args:
        tanggal: datetime.date object atau string format 'YYYY-MM-DD'
        province_id: int (15=Jawa Timur, default)
        filter_beras: bool (True=hanya Beras Premium & Medium, False=semua data)

    Returns:
        tuple: (success: bool, df_konsumen: pd.DataFrame, df_produsen: pd.DataFrame, message: str)

    Example:
        >>> import datetime
        >>> success, df_konsumen, df_produsen, msg = get_bapanas_konsumen_produsen(datetime.date.today(), filter_beras=True)
        >>> if success:
        >>>     print("Konsumen:", len(df_konsumen), "baris")
        >>>     print("Produsen:", len(df_produsen), "baris")
    """
    # Ambil data konsumen (level_harga_id=3)
    success_konsumen, df_konsumen, msg_konsumen = get_bapanas_dataframe(
        tanggal, level_harga_id=3, province_id=province_id, filter_beras=filter_beras
    )

    # Ambil data produsen (level_harga_id=1)
    success_produsen, df_produsen, msg_produsen = get_bapanas_dataframe(
        tanggal, level_harga_id=1, province_id=province_id, filter_beras=filter_beras
    )

    # Evaluasi hasil
    if success_konsumen and success_produsen:
        message = f"✅ Berhasil mengambil keduanya.\nKonsumen: {len(df_konsumen)} baris\nProdusen: {len(df_produsen)} baris"
        return True, df_konsumen, df_produsen, message
    elif success_konsumen:
        message = f"⚠️ Hanya konsumen berhasil ({len(df_konsumen)} baris). Produsen gagal: {msg_produsen}"
        return True, df_konsumen, pd.DataFrame(), message
    elif success_produsen:
        message = f"⚠️ Hanya produsen berhasil ({len(df_produsen)} baris). Konsumen gagal: {msg_konsumen}"
        return True, pd.DataFrame(), df_produsen, message
    else:
        message = f"❌ Keduanya gagal.\nKonsumen: {msg_konsumen}\nProdusen: {msg_produsen}"
        return False, pd.DataFrame(), pd.DataFrame(), message


def get_bapanas_periode(start_date, end_date, level_harga_id=3, province_id=15, filter_beras=False):
    """
    Mengambil data BAPANAS untuk rentang tanggal dan menggabungkannya
    menjadi satu DataFrame.

    Args:
        start_date: datetime.date object atau string 'YYYY-MM-DD'
        end_date: datetime.date object atau string 'YYYY-MM-DD'
        level_harga_id: int (3=konsumen, 1=produsen, dst)
        province_id: int (15=Jawa Timur, default)
        filter_beras: bool (True=hanya Beras Premium & Medium, False=semua data)

    Returns:
        tuple: (success: bool, dataframe: pd.DataFrame, message: str)

    Example:
        >>> success, df, msg = get_bapanas_periode('2024-01-01', '2024-01-07', filter_beras=True)
        >>> if success:
        >>>     print(f"Total data: {len(df)} baris")
    """
    # Konversi string ke datetime.date jika perlu
    if isinstance(start_date, str):
        start_date = datetime.datetime.strptime(start_date, '%Y-%m-%d').date()
    if isinstance(end_date, str):
        end_date = datetime.datetime.strptime(end_date, '%Y-%m-%d').date()

    if start_date > end_date:
        return False, pd.DataFrame(), "Tanggal mulai tidak boleh lebih besar dari tanggal akhir"

    all_data = []
    total_days = (end_date - start_date).days + 1
    failed_dates = []

    for i in range(total_days):
        current_date = start_date + datetime.timedelta(days=i)
        success, df, msg = get_bapanas_dataframe(current_date, level_harga_id, province_id, filter_beras)

        if success and not df.empty:
            # Tambahkan kolom tanggal untuk tracking
            df['tanggal_scrape'] = current_date
            all_data.append(df)
        else:
            failed_dates.append(current_date.strftime('%Y-%m-%d'))

    if not all_data:
        return False, pd.DataFrame(), "Tidak ada data berhasil diambil"

    # Gabungkan semua dataframe
    result_df = pd.concat(all_data, ignore_index=True)

    filter_msg = " (Beras Premium & Medium)" if filter_beras else ""
    message = f"Berhasil mengambil {len(all_data)} dari {total_days} hari. Total {len(result_df)} baris data{filter_msg}."
    if failed_dates:
        message += f"\nGagal: {', '.join(failed_dates[:5])}"
        if len(failed_dates) > 5:
            message += f" ... (+{len(failed_dates)-5} lainnya)"

    return True, result_df, message


def get_bapanas_periode_konsumen_produsen(start_date, end_date, province_id=15, filter_beras=False):
    """
    Mengambil data konsumen DAN produsen untuk rentang tanggal.

    Args:
        start_date: datetime.date object atau string 'YYYY-MM-DD'
        end_date: datetime.date object atau string 'YYYY-MM-DD'
        province_id: int (15=Jawa Timur, default)
        filter_beras: bool (True=hanya Beras Premium & Medium, False=semua data)

    Returns:
        tuple: (success: bool, df_konsumen: pd.DataFrame, df_produsen: pd.DataFrame, message: str)

    Example:
        >>> success, df_k, df_p, msg = get_bapanas_periode_konsumen_produsen('2024-01-01', '2024-01-07', filter_beras=True)
        >>> if success:
        >>>     print("Konsumen:", len(df_k), "baris")
        >>>     print("Produsen:", len(df_p), "baris")
    """
    # Ambil data konsumen untuk periode
    success_konsumen, df_konsumen, msg_konsumen = get_bapanas_periode(
        start_date, end_date, level_harga_id=3, province_id=province_id, filter_beras=filter_beras
    )

    # Ambil data produsen untuk periode
    success_produsen, df_produsen, msg_produsen = get_bapanas_periode(
        start_date, end_date, level_harga_id=1, province_id=province_id, filter_beras=filter_beras
    )

    # Evaluasi hasil
    if success_konsumen and success_produsen:
        message = f"✅ Berhasil mengambil keduanya.\nKonsumen: {len(df_konsumen)} baris\nProdusen: {len(df_produsen)} baris"
        return True, df_konsumen, df_produsen, message
    elif success_konsumen:
        message = f"⚠️ Hanya konsumen berhasil ({len(df_konsumen)} baris).\nProdusen: {msg_produsen}"
        return True, df_konsumen, pd.DataFrame(), message
    elif success_produsen:
        message = f"⚠️ Hanya produsen berhasil ({len(df_produsen)} baris).\nKonsumen: {msg_konsumen}"
        return True, pd.DataFrame(), df_produsen, message
    else:
        message = f"❌ Keduanya gagal.\nKonsumen: {msg_konsumen}\nProdusen: {msg_produsen}"
        return False, pd.DataFrame(), pd.DataFrame(), message

def deduplicate_columns(df):
    """Rename duplicate columns by appending _1, _2, etc."""
    cols = pd.Series(df.columns)
    for dup in cols[cols.duplicated()].unique():
        mask = cols == dup
        cols[mask] = [f"{dup}_{i}" if i != 0 else dup for i, _ in enumerate(mask[mask].index)]
    df.columns = cols
    return df

start_date = datetime.date(2025, 12, 19)
end_date   = datetime.date(2026, 2, 19)

# Buat folder output
output_folder = "data_harian"
os.makedirs(output_folder, exist_ok=True)

current_date = start_date

while current_date <= end_date:
    
    print(f"Ambil data: {current_date}")
    
    success, df_konsumen, df_produsen, message = get_bapanas_konsumen_produsen(
        current_date, filter_beras=False
    )
    
    if success:
        combined_df = pd.DataFrame()
        
        if df_konsumen is not None and not df_konsumen.empty:
            df_konsumen = deduplicate_columns(df_konsumen)
            df_konsumen["tipe"] = "konsumen"
            combined_df = pd.concat([combined_df, df_konsumen], ignore_index=True)
        
        if df_produsen is not None and not df_produsen.empty:
            df_produsen = deduplicate_columns(df_produsen)
            df_produsen["tipe"] = "produsen"
            combined_df = pd.concat([combined_df, df_produsen], ignore_index=True)
        
        if not combined_df.empty:
            file_name = f"{current_date}.xlsx"
            file_path = os.path.join(output_folder, file_name)
            combined_df.to_excel(file_path, index=False)
            print(f"✔ Exported: {file_name}")
        else:
            print("⚠ Data kosong")
    
    else:
        print("✖ Gagal:", message)
    
    current_date += datetime.timedelta(days=1)

print("Selesai ✅")

Ambil data: 2025-12-19
✔ Exported: 2025-12-19.xlsx
Ambil data: 2025-12-20
✔ Exported: 2025-12-20.xlsx
Ambil data: 2025-12-21
✔ Exported: 2025-12-21.xlsx
Ambil data: 2025-12-22
✔ Exported: 2025-12-22.xlsx
Ambil data: 2025-12-23
✔ Exported: 2025-12-23.xlsx
Ambil data: 2025-12-24
✔ Exported: 2025-12-24.xlsx
Ambil data: 2025-12-25
✔ Exported: 2025-12-25.xlsx
Ambil data: 2025-12-26
✔ Exported: 2025-12-26.xlsx
Ambil data: 2025-12-27
✔ Exported: 2025-12-27.xlsx
Ambil data: 2025-12-28
✔ Exported: 2025-12-28.xlsx
Ambil data: 2025-12-29
✔ Exported: 2025-12-29.xlsx
Ambil data: 2025-12-30
✔ Exported: 2025-12-30.xlsx
Ambil data: 2025-12-31
✔ Exported: 2025-12-31.xlsx
Ambil data: 2026-01-01
✔ Exported: 2026-01-01.xlsx
Ambil data: 2026-01-02
✔ Exported: 2026-01-02.xlsx
Ambil data: 2026-01-03
✔ Exported: 2026-01-03.xlsx
Ambil data: 2026-01-04
✔ Exported: 2026-01-04.xlsx
Ambil data: 2026-01-05
✔ Exported: 2026-01-05.xlsx
Ambil data: 2026-01-06
✔ Exported: 2026-01-06.xlsx
Ambil data: 2026-01-07
✔ Export

In [1]:
import requests
import pandas as pd
from io import BytesIO
import datetime
from dateutil.relativedelta import relativedelta
import time


# ── Fungsi dasar ───────────────────────────────────────────────────────────────
def get_bapanas_dataframe(tanggal, level_harga_id=3, province_id=15, filter_beras=False):
    if isinstance(tanggal, str):
        tanggal = datetime.datetime.strptime(tanggal, '%Y-%m-%d').date()

    formatted = tanggal.strftime("%d/%m/%Y")
    period_param = f"{formatted}%20-%20{formatted}"
    url = f"https://api-panelhargav2.badanpangan.go.id/harga-pangan-table-province/export?province_id={province_id}&period_date={period_param}&level_harga_id={level_harga_id}"

    try:
        response = requests.get(url, timeout=30)
        if response.status_code == 200:
            df = pd.read_excel(BytesIO(response.content))
            if df.empty:
                return False, df, "Data kosong"

            if filter_beras:
                kolom_identitas = [col for col in df.columns if any(k in col.lower() for k in ['no', 'kota', 'kabupaten', 'wilayah', 'daerah', 'provinsi'])]
                kolom_beras = [col for col in df.columns if 'Beras Premium' in col or 'Beras Medium' in col]
                if not kolom_beras:
                    return False, pd.DataFrame(), "Kolom beras tidak ditemukan"
                df = df[kolom_identitas + kolom_beras]

            return True, df, f"Berhasil {len(df)} baris"
        else:
            return False, pd.DataFrame(), f"HTTP {response.status_code}"
    except Exception as e:
        return False, pd.DataFrame(), f"Error: {str(e)}"


# ── Generate rentang bulan ─────────────────────────────────────────────────────
def generate_monthly_ranges(start: datetime.date, end: datetime.date):
    ranges = []
    current = start.replace(day=1)
    while current <= end:
        month_start = current
        month_end = (current + relativedelta(months=1)) - datetime.timedelta(days=1)
        month_end = min(month_end, end)
        ranges.append((month_start, month_end))
        current += relativedelta(months=1)
    return ranges


# ── Fungsi utama: loop per bulan ───────────────────────────────────────────────
def get_bapanas_periode_bulanan(start_date, end_date, level_harga_id=3, province_id=15, filter_beras=False):
    if isinstance(start_date, str):
        start_date = datetime.datetime.strptime(start_date, '%Y-%m-%d').date()
    if isinstance(end_date, str):
        end_date = datetime.datetime.strptime(end_date, '%Y-%m-%d').date()

    monthly_ranges = generate_monthly_ranges(start_date, end_date)
    total_months = len(monthly_ranges)
    print(f"Total bulan yang akan diproses: {total_months} bulan")
    print(f"Periode: {start_date} s/d {end_date}\n")

    all_data = []
    failed_dates = []
    total_success = 0
    total_days = 0

    for idx, (month_start, month_end) in enumerate(monthly_ranges, 1):
        print(f"[{idx}/{total_months}] Proses bulan: {month_start.strftime('%B %Y')} ({month_start} s/d {month_end})")

        days_in_month = (month_end - month_start).days + 1
        month_success = 0
        month_failed  = 0

        for i in range(days_in_month):
            current_date = month_start + datetime.timedelta(days=i)

            # ← DIHAPUS: tidak ada skip Sabtu/Minggu, semua hari diproses

            total_days += 1
            success, df, msg = get_bapanas_dataframe(current_date, level_harga_id, province_id, filter_beras)

            if success and not df.empty:
                df['tanggal'] = pd.to_datetime(current_date)
                df['bulan']   = current_date.strftime('%Y-%m-%d')
                df['hari']    = current_date.strftime('%A')   # ← tambah kolom nama hari
                all_data.append(df)
                month_success += 1
                total_success += 1
            else:
                month_failed += 1
                failed_dates.append(current_date.strftime('%Y-%m-%d'))
                print(f"    ⚠ Tidak ada data: {current_date} ({current_date.strftime('%A')})")

            time.sleep(0.2)

        status = "✓" if month_failed == 0 else "⚠"
        print(f"  {status} Bulan selesai: {month_success}/{days_in_month} hari berhasil"
              + (f" | Gagal: {month_failed} hari" if month_failed > 0 else "") + "\n")
        time.sleep(1)

    if not all_data:
        return False, pd.DataFrame(), "Tidak ada data berhasil diambil"

    result_df = pd.concat(all_data, ignore_index=True)
    result_df['tanggal'] = pd.to_datetime(result_df['tanggal'])

    message = (
        f"✅ Selesai!\n"
        f"   Total hari diproses : {total_days}\n"
        f"   Berhasil            : {total_success} hari\n"
        f"   Gagal               : {len(failed_dates)} hari\n"
        f"   Total baris data    : {len(result_df)}"
    )
    if failed_dates:
        message += f"\n   Contoh gagal        : {', '.join(failed_dates[:5])}"
        if len(failed_dates) > 5:
            message += f" ... (+{len(failed_dates)-5} lainnya)"

    return True, result_df, message


# ── Jalankan ───────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    START_DATE = "2026-02-17"
    END_DATE   = "2026-02-18"

    # Ambil data KONSUMEN
    print("=" * 60)
    print("Mengambil Data KONSUMEN")
    print("=" * 60)
    success_k, df_konsumen, msg_k = get_bapanas_periode_bulanan(
        START_DATE, END_DATE, level_harga_id=3, province_id=15, filter_beras=True
    )
    print(msg_k)

    # Ambil data PRODUSEN
    print("\n" + "=" * 60)
    print("Mengambil Data PRODUSEN")
    print("=" * 60)
    success_p, df_produsen, msg_p = get_bapanas_periode_bulanan(
        START_DATE, END_DATE, level_harga_id=1, province_id=15, filter_beras=True
    )
    print(msg_p)

    # Simpan ke Excel (2 sheet)
    if success_k or success_p:
        output_file = "2024-2026.xlsx"
        with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
            if success_k and not df_konsumen.empty:
                df_konsumen.to_excel(writer, sheet_name="Konsumen", index=False)
            if success_p and not df_produsen.empty:
                df_produsen.to_excel(writer, sheet_name="Produsen", index=False)
        print(f"\n✅ File disimpan: {output_file}")

Mengambil Data KONSUMEN
Total bulan yang akan diproses: 1 bulan
Periode: 2026-02-17 s/d 2026-02-18

[1/1] Proses bulan: February 2026 (2026-02-01 s/d 2026-02-18)
    ⚠ Tidak ada data: 2026-02-01 (Sunday)
    ⚠ Tidak ada data: 2026-02-02 (Monday)
    ⚠ Tidak ada data: 2026-02-03 (Tuesday)
    ⚠ Tidak ada data: 2026-02-04 (Wednesday)
    ⚠ Tidak ada data: 2026-02-05 (Thursday)
    ⚠ Tidak ada data: 2026-02-06 (Friday)
    ⚠ Tidak ada data: 2026-02-07 (Saturday)
    ⚠ Tidak ada data: 2026-02-08 (Sunday)
    ⚠ Tidak ada data: 2026-02-09 (Monday)


KeyboardInterrupt: 

# SP2KP Alternatif scraping

In [ ]:
import requests
import pandas as pd

url = "https://api-sp2kp.kemendag.go.id/report/api/average-price/export-area-daily-json"

headers = {
    "User-Agent": "Mozilla/5.0",
    "Origin": "https://sp2kp.kemendag.go.id",
    "Referer": "https://sp2kp.kemendag.go.id/",
    "Accept": "application/json"
}

payload = {
    "start_date": "2026-02-12",
    "end_date": "2026-02-12",
    "level": 2,
    "variant_ids": "52,51",   # STRING dipisah koma
    "kode_provinsi": 35,
    "kode_kab_kota": 3501,
    "pasar_id": "",
    "skip_sat_sun": "true",
    "tipe_komoditas": ""
}

response = requests.post(url, headers=headers, data=payload)

print(response.status_code)

data = response.json()
print(data)


# Mengakses daftarHarga dan membuat DataFrame
df_beras = pd.json_normalize(data['data'], 'daftarHarga', ['kuantitas', 'order', 'satuan', 'variant', 'variant_id'], errors='ignore')

# Menampilkan DataFrame
print(df_beras)

200
{'status': 'success', 'message': 'Success retrieving record', 'data': [{'daftarHarga': [{'date': '2026-02-12', 'harga': 13267}], 'kuantitas': 1, 'order': 101, 'satuan': 'kg', 'variant': 'Beras Medium', 'variant_id': 52}, {'daftarHarga': [{'date': '2026-02-12', 'harga': 14900}], 'kuantitas': 1, 'order': 102, 'satuan': 'kg', 'variant': 'Beras Premium', 'variant_id': 51}]}
         date  harga kuantitas order satuan        variant variant_id
0  2026-02-12  13267         1   101     kg   Beras Medium         52
1  2026-02-12  14900         1   102     kg  Beras Premium         51


In [3]:
import requests
import pandas as pd

headers = {
    "User-Agent": "Mozilla/5.0",
    "Origin": "https://sp2kp.kemendag.go.id",
    "Referer": "https://sp2kp.kemendag.go.id/"
}

url_prov = "https://api-sp2kp.kemendag.go.id/master/api/wilayah/provinsi?take=9999"

response = requests.get(url_prov, headers=headers)
prov_data = response.json()

df_prov = pd.DataFrame(prov_data["data"])  # biasanya ada key "data"
print(df_prov)


   kode_provinsi               nama_provinsi nama_singkat_provinsi wilayah
0             11                        Aceh                  Aceh   Barat
1             12              Sumatera Utara                 Sumut   Barat
2             13              Sumatera Barat                Sumbar   Barat
3             14                        Riau                  Riau   Barat
4             15                       Jambi                 Jambi   Barat
5             16            Sumatera Selatan                Sumsel   Barat
6             17                    Bengkulu              Bengkulu   Barat
7             18                     Lampung               Lampung   Barat
8             19   Kepulauan Bangka Belitung                 Babel   Barat
9             21              Kepulauan Riau                 Kepri   Barat
10            31                 DKI Jakarta           DKI Jakarta   Barat
11            32                  Jawa Barat                 Jabar  Tengah
12            33         

In [4]:
url_pasar = "https://api-sp2kp.kemendag.go.id/master/api/pasar"

params = {
    "take": 9999,
    "kode_provinsi": 35,
    "tipe_komoditas_id": 1,
    "is_nasional": "true"
}

response = requests.get(url_pasar, headers=headers, params=params)
data = response.json()

print("Total pasar:", data["totalCount"])

df = pd.DataFrame(data["data"])
print(df[["id", "nama", "kode_kab_kota"]])


Total pasar: 40
      id                       nama kode_kab_kota
0     39               Pasar Porong          3515
1     42            Pasar Anom Baru          3529
2     47          Pasar Pucang Anom          3578
3    187             Pasar Kepanjen          3507
4    395              Pasar Minulyo          3501
5    398     Pasar Basah Trenggalek          3503
6    399        Pasar Rakyat Ngunut          3504
7    400             Pasar Kanigoro          3505
8    401             Pasar Pamenang          3506
9    405           Pasar Banyuwangi          3510
10   406      Pasar Induk Bondowoso          3511
11   407         Pasar Mimbaan Baru          3512
12   396        Pasar Legi Ponorogo          3502
13   420             Pasar Semampir          3513
14   421               Pasar Bangil          3514
15   422        Pasar Raya Mojosari          3516
16   423          Pasar Pon Jombang          3517
17   424             Pasar Sukomoro          3518
18   425         Pasar Umum Caruba

In [10]:
url = "https://api-sp2kp.kemendag.go.id/master/api/wilayah/kab-kota/35"
response = requests.get(url, headers=headers)

print(response.status_code)
data = response.json()

df_kota = pd.DataFrame(data["data"])  # biasanya ada key "data"
print(df_kota)

200
   kode_kab_kota     nama_kab_kota
0           3501      Kab. Pacitan
1           3502     Kab. Ponorogo
2           3503   Kab. Trenggalek
3           3504  Kab. Tulungagung
4           3505       Kab. Blitar
5           3506       Kab. Kediri
6           3507       Kab. Malang
7           3508     Kab. Lumajang
8           3509       Kab. Jember
9           3510   Kab. Banyuwangi
10          3511    Kab. Bondowoso
11          3512    Kab. Situbondo
12          3513  Kab. Probolinggo
13          3514     Kab. Pasuruan
14          3515     Kab. Sidoarjo
15          3516    Kab. Mojokerto
16          3517      Kab. Jombang
17          3518      Kab. Nganjuk
18          3519       Kab. Madiun
19          3520      Kab. Magetan
20          3521        Kab. Ngawi
21          3522   Kab. Bojonegoro
22          3523        Kab. Tuban
23          3524     Kab. Lamongan
24          3525       Kab. Gresik
25          3526    Kab. Bangkalan
26          3527      Kab. Sampang
27          3528

In [ ]:
import requests
import pandas as pd
from datetime import datetime
import time

# ===============================
# SETUP
# ===============================

today = datetime.today().strftime("%Y-%m-%d")

headers = {
    "User-Agent": "Mozilla/5.0",
    "Origin": "https://sp2kp.kemendag.go.id",
    "Referer": "https://sp2kp.kemendag.go.id/",
    "Accept": "application/json"
}

kode_provinsi = 35
variant_ids = "52,51"

# ===============================
# 1️⃣ AMBIL SEMUA PASAR
# ===============================

url_pasar = "https://api-sp2kp.kemendag.go.id/master/api/pasar"

params = {
    "take": 9999,
    "kode_provinsi": kode_provinsi,
    "tipe_komoditas_id": 1,
    "is_nasional": "true"
}

response = requests.get(url_pasar, headers=headers, params=params)

if response.status_code != 200:
    raise Exception("Gagal mengambil data pasar")

data = response.json()

df_pasar = pd.DataFrame(data.get("data", []))

print("Total pasar:", len(df_pasar))

# ===============================
# 2️⃣ LOOP AMBIL HARGA TIAP PASAR
# ===============================

url_harga = "https://api-sp2kp.kemendag.go.id/report/api/average-price/export-area-daily-json"

all_data = []

for index, row in df_pasar.iterrows():

    payload = {
        "start_date": today,
        "end_date": today,
        "level": 2,
        "variant_ids": variant_ids,
        "kode_provinsi": kode_provinsi,
        "kode_kab_kota": row["kode_kab_kota"],
        "pasar_id": row["id"],
        "skip_sat_sun": "true",
        "tipe_komoditas": ""
    }

    try:
        r = requests.post(url_harga, headers=headers, data=payload, timeout=10)

        if r.status_code != 200:
            print(f"Gagal ambil harga {row['nama']}")
            continue

        result = r.json()

        # ===============================
        # HANDLE SEMUA KEMUNGKINAN FORMAT
        # ===============================

        # Jika result dict dan ada key 'data'
        if isinstance(result, dict):
            result = result.get("data", [])

        # Jika result string (kadang API return string kosong)
        if isinstance(result, str):
            print(f"Tidak ada data di {row['nama']}")
            continue

        # Jika result list
        if isinstance(result, list):
            for item in result:
                if isinstance(item, dict):
                    item_copy = item.copy()
                    item_copy["nama_pasar"] = row["nama"]
                    item_copy["kode_kab_kota"] = row["kode_kab_kota"]
                    all_data.append(item_copy)

        print(f"✓ Sukses: {row['nama']}")

        time.sleep(0.3)  # biar tidak dianggap spam API-

    except Exception as e:
        print(f"Error di {row['nama']}:", e)


# ===============================
# 3️⃣ GABUNGKAN KE DATAFRAME
# ===============================

df_all = pd.DataFrame(all_data)

print("\nTotal data terkumpul:", len(df_all))
print(df_all.head())

# ===============================
# 4️⃣ SIMPAN KE CSV (OPSIONAL)
# ===============================

if not df_all.empty:
    filename = f"harga_sp2kp_{today}.csv"
    df_all.to_csv(filename, index=False)
    print(f"\nData disimpan ke {filename}")
else:
    print("\nTidak ada data yang berhasil dikumpulkan.")


Total pasar: 40
✓ Sukses: Pasar Porong
✓ Sukses: Pasar Anom Baru
✓ Sukses: Pasar Pucang Anom
✓ Sukses: Pasar Kepanjen
✓ Sukses: Pasar Minulyo
✓ Sukses: Pasar Basah Trenggalek
✓ Sukses: Pasar Rakyat Ngunut
✓ Sukses: Pasar Kanigoro
✓ Sukses: Pasar Pamenang
✓ Sukses: Pasar Banyuwangi
✓ Sukses: Pasar Induk Bondowoso
✓ Sukses: Pasar Mimbaan Baru
✓ Sukses: Pasar Legi Ponorogo
✓ Sukses: Pasar Semampir
✓ Sukses: Pasar Bangil
✓ Sukses: Pasar Raya Mojosari
✓ Sukses: Pasar Pon Jombang
✓ Sukses: Pasar Sukomoro
✓ Sukses: Pasar Umum Caruban
✓ Sukses: Pasar Sayur 1 Magetan
✓ Sukses: Pasar Walikukun
✓ Sukses: Pasar Baru Tuban
✓ Sukses: Pasar Ki Lemah Duwur
✓ Sukses: Pasar Srimangunan
✓ Sukses: Pasar Kolpajung
✓ Sukses: Pasar Setono Betek
✓ Sukses: Pasar Templek
✓ Sukses: Pasar Bunulrejo
✓ Sukses: Pasar Besar Kota Pasuruan
✓ Sukses: Pasar Tanjung Anyar
✓ Sukses: Pasar Besar Kota Madiun
✓ Sukses: Pasar Baru Gresik
✓ Sukses: Pasar Sidoharjo
✓ Sukses: Pasar Genteng
✓ Sukses: Pasar Among Tani
✓ Sukses: Pas

In [ ]:
import requests
import pandas as pd

# URL API
url = "https://api-sp2kp.kemendag.go.id/report/api/average-price/export-area-daily-json"

# Headers untuk permintaan API
headers = {
    "User-Agent": "Mozilla/5.0",
    "Origin": "https://sp2kp.kemendag.go.id",
    "Referer": "https://sp2kp.kemendag.go.id/",
    "Accept": "application/json"
}

# Daftar kota-kota di Jawa Timur beserta kode kota
kota_jatim = [
    {"kode_kab_kota": 3501, "nama_kab_kota": "Kab. Pacitan"},
    {"kode_kab_kota": 3502, "nama_kab_kota": "Kab. Ponorogo"},
    {"kode_kab_kota": 3503, "nama_kab_kota": "Kab. Trenggalek"},
    {"kode_kab_kota": 3504, "nama_kab_kota": "Kab. Tulungagung"},
    {"kode_kab_kota": 3505, "nama_kab_kota": "Kab. Blitar"},
    {"kode_kab_kota": 3506, "nama_kab_kota": "Kab. Kediri"},
    {"kode_kab_kota": 3507, "nama_kab_kota": "Kab. Malang"},
    {"kode_kab_kota": 3508, "nama_kab_kota": "Kab. Lumajang"},
    {"kode_kab_kota": 3509, "nama_kab_kota": "Kab. Jember"},
    {"kode_kab_kota": 3510, "nama_kab_kota": "Kab. Banyuwangi"},
    {"kode_kab_kota": 3511, "nama_kab_kota": "Kab. Bondowoso"},
    {"kode_kab_kota": 3512, "nama_kab_kota": "Kab. Situbondo"},
    {"kode_kab_kota": 3513, "nama_kab_kota": "Kab. Probolinggo"},
    {"kode_kab_kota": 3514, "nama_kab_kota": "Kab. Pasuruan"},
    {"kode_kab_kota": 3515, "nama_kab_kota": "Kab. Sidoarjo"},
    {"kode_kab_kota": 3516, "nama_kab_kota": "Kab. Mojokerto"},
    {"kode_kab_kota": 3517, "nama_kab_kota": "Kab. Jombang"},
    {"kode_kab_kota": 3518, "nama_kab_kota": "Kab. Nganjuk"},
    {"kode_kab_kota": 3519, "nama_kab_kota": "Kab. Madiun"},
    {"kode_kab_kota": 3520, "nama_kab_kota": "Kab. Magetan"},
    {"kode_kab_kota": 3521, "nama_kab_kota": "Kab. Ngawi"},
    {"kode_kab_kota": 3522, "nama_kab_kota": "Kab. Bojonegoro"},
    {"kode_kab_kota": 3523, "nama_kab_kota": "Kab. Tuban"},
    {"kode_kab_kota": 3524, "nama_kab_kota": "Kab. Lamongan"},
    {"kode_kab_kota": 3525, "nama_kab_kota": "Kab. Gresik"},
    {"kode_kab_kota": 3526, "nama_kab_kota": "Kab. Bangkalan"},
    {"kode_kab_kota": 3527, "nama_kab_kota": "Kab. Sampang"},
    {"kode_kab_kota": 3528, "nama_kab_kota": "Kab. Pamekasan"},
    {"kode_kab_kota": 3529, "nama_kab_kota": "Kab. Sumenep"},
    {"kode_kab_kota": 3571, "nama_kab_kota": "Kota Kediri"},
    {"kode_kab_kota": 3572, "nama_kab_kota": "Kota Blitar"},
    {"kode_kab_kota": 3573, "nama_kab_kota": "Kota Malang"},
    {"kode_kab_kota": 3574, "nama_kab_kota": "Kota Probolinggo"},
    {"kode_kab_kota": 3575, "nama_kab_kota": "Kota Pasuruan"},
    {"kode_kab_kota": 3576, "nama_kab_kota": "Kota Mojokerto"},
    {"kode_kab_kota": 3577, "nama_kab_kota": "Kota Madiun"},
    {"kode_kab_kota": 3578, "nama_kab_kota": "Kota Surabaya"},
    {"kode_kab_kota": 3579, "nama_kab_kota": "Kota Batu"}
]

# Mengambil data harga untuk setiap kota di Jawa Timur
all_data = []  # Untuk menyimpan semua data harga

# Loop untuk setiap kota
for kota in kota_jatim:
    payload = {
        "start_date": "2024-02-01",  # Tanggal mulai (1 Februari 2026)
        "end_date": "2026-02-28",  # Tanggal selesai (28 Februari 2026)
        "level": 2,
        "variant_ids": "52,51",   # ID variant (Beras Medium dan Premium)
        "kode_provinsi": 35,  # Kode provinsi Jawa Timur
        "kode_kab_kota": kota["kode_kab_kota"],  # Kode kota
        "pasar_id": "",
        "skip_sat_sun": "true",
        "tipe_komoditas": ""
    }
    
    # Mengirim request
    response = requests.post(url, headers=headers, data=payload)
    
    # Mengecek status response
    if response.status_code == 200:
        data = response.json()
        
        # Memastikan data berisi informasi harga untuk kedua varian
        if 'data' in data:
            for entry in data['data']:
                # Menampilkan semua data untuk memeriksa harga per varian
                print(entry)  # Cek entry
                # Menyaring harga berdasarkan varian yang ada
                if 'daftarHarga' in entry:
                    for harga_entry in entry['daftarHarga']:
                        # Mengambil harga dan menambah nama kota serta varian
                        df_beras = pd.json_normalize(harga_entry)
                        df_beras['kota'] = kota['nama_kab_kota']  # Menambahkan kolom kota
                        df_beras['variant_id'] = entry['variant_id']  # Menambahkan kolom variant_id
                        df_beras['variant'] = entry['variant']  # Menambahkan kolom variant
                        all_data.append(df_beras)  # Menambahkan data ke list
        else:
            print(f"Data tidak ditemukan untuk kota {kota['nama_kab_kota']}")
    else:
        print(f"Failed to fetch data for {kota['nama_kab_kota']}")

# Menggabungkan semua data menjadi satu DataFrame
if all_data:
    final_df = pd.concat(all_data, ignore_index=True)
    print(final_df)
    final_df.to_excel("harga_beras_jatim_feb2026.xlsx", index=False)  # Menyimpan ke Excel


TypeError: 'NoneType' object is not iterable

In [1]:
import requests
import pandas as pd
from datetime import date
from dateutil.relativedelta import relativedelta
import time

# URL API
url = "https://api-sp2kp.kemendag.go.id/report/api/average-price/export-area-daily-json"

headers = {
    "User-Agent": "Mozilla/5.0",
    "Origin": "https://sp2kp.kemendag.go.id",
    "Referer": "https://sp2kp.kemendag.go.id/",
    "Accept": "application/json"
}

kota_jatim = [
    {"kode_kab_kota": 3501, "nama_kab_kota": "Kab. Pacitan"},
    {"kode_kab_kota": 3502, "nama_kab_kota": "Kab. Ponorogo"},
    {"kode_kab_kota": 3503, "nama_kab_kota": "Kab. Trenggalek"},
    {"kode_kab_kota": 3504, "nama_kab_kota": "Kab. Tulungagung"},
    {"kode_kab_kota": 3505, "nama_kab_kota": "Kab. Blitar"},
    {"kode_kab_kota": 3506, "nama_kab_kota": "Kab. Kediri"},
    {"kode_kab_kota": 3507, "nama_kab_kota": "Kab. Malang"},
    {"kode_kab_kota": 3508, "nama_kab_kota": "Kab. Lumajang"},
    {"kode_kab_kota": 3509, "nama_kab_kota": "Kab. Jember"},
    {"kode_kab_kota": 3510, "nama_kab_kota": "Kab. Banyuwangi"},
    {"kode_kab_kota": 3511, "nama_kab_kota": "Kab. Bondowoso"},
    {"kode_kab_kota": 3512, "nama_kab_kota": "Kab. Situbondo"},
    {"kode_kab_kota": 3513, "nama_kab_kota": "Kab. Probolinggo"},
    {"kode_kab_kota": 3514, "nama_kab_kota": "Kab. Pasuruan"},
    {"kode_kab_kota": 3515, "nama_kab_kota": "Kab. Sidoarjo"},
    {"kode_kab_kota": 3516, "nama_kab_kota": "Kab. Mojokerto"},
    {"kode_kab_kota": 3517, "nama_kab_kota": "Kab. Jombang"},
    {"kode_kab_kota": 3518, "nama_kab_kota": "Kab. Nganjuk"},
    {"kode_kab_kota": 3519, "nama_kab_kota": "Kab. Madiun"},
    {"kode_kab_kota": 3520, "nama_kab_kota": "Kab. Magetan"},
    {"kode_kab_kota": 3521, "nama_kab_kota": "Kab. Ngawi"},
    {"kode_kab_kota": 3522, "nama_kab_kota": "Kab. Bojonegoro"},
    {"kode_kab_kota": 3523, "nama_kab_kota": "Kab. Tuban"},
    {"kode_kab_kota": 3524, "nama_kab_kota": "Kab. Lamongan"},
    {"kode_kab_kota": 3525, "nama_kab_kota": "Kab. Gresik"},
    {"kode_kab_kota": 3526, "nama_kab_kota": "Kab. Bangkalan"},
    {"kode_kab_kota": 3527, "nama_kab_kota": "Kab. Sampang"},
    {"kode_kab_kota": 3528, "nama_kab_kota": "Kab. Pamekasan"},
    {"kode_kab_kota": 3529, "nama_kab_kota": "Kab. Sumenep"},
    {"kode_kab_kota": 3571, "nama_kab_kota": "Kota Kediri"},
    {"kode_kab_kota": 3572, "nama_kab_kota": "Kota Blitar"},
    {"kode_kab_kota": 3573, "nama_kab_kota": "Kota Malang"},
    {"kode_kab_kota": 3574, "nama_kab_kota": "Kota Probolinggo"},
    {"kode_kab_kota": 3575, "nama_kab_kota": "Kota Pasuruan"},
    {"kode_kab_kota": 3576, "nama_kab_kota": "Kota Mojokerto"},
    {"kode_kab_kota": 3577, "nama_kab_kota": "Kota Madiun"},
    {"kode_kab_kota": 3578, "nama_kab_kota": "Kota Surabaya"},
    {"kode_kab_kota": 3579, "nama_kab_kota": "Kota Batu"}
]

# ── Generate daftar rentang bulan ──────────────────────────────────────────────
def generate_monthly_ranges(start: date, end: date):
    """Menghasilkan list tuple (start_date, end_date) per bulan."""
    ranges = []
    current = start.replace(day=1)
    while current <= end:
        month_start = current
        # Akhir bulan = awal bulan berikutnya - 1 hari
        month_end = (current + relativedelta(months=1)) - relativedelta(days=1)
        # Batasi end_date agar tidak melebihi tanggal akhir yang ditentukan
        month_end = min(month_end, end)
        ranges.append((month_start.strftime("%Y-%m-%d"), month_end.strftime("%Y-%m-%d")))
        current += relativedelta(months=1)
    return ranges

START_DATE = date(2024, 1, 1)
END_DATE   = date(2026, 2, 18)

monthly_ranges = generate_monthly_ranges(START_DATE, END_DATE)
print(f"Total rentang bulan: {len(monthly_ranges)}")
# Contoh output: [('2024-01-01','2024-01-31'), ('2024-02-01','2024-02-29'), ...]

# ── Loop utama ─────────────────────────────────────────────────────────────────
all_data = []

for kota in kota_jatim:
    for start_date, end_date in monthly_ranges:
        print(f"Mengambil: {kota['nama_kab_kota']} | {start_date} s/d {end_date}")

        payload = {
            "start_date": start_date,
            "end_date": end_date,
            "level": 1,
            "variant_ids": "52,51",
            "kode_provinsi": 35,
            "kode_kab_kota": kota["kode_kab_kota"],
            "pasar_id": "",
            "skip_sat_sun": "true",
            "tipe_komoditas": ""
        }

        try:
            response = requests.post(url, headers=headers, data=payload, timeout=30)

            if response.status_code == 200:
                data = response.json()
                if 'data' in data:
                    for entry in (data.get('data') or []):   # ← jika None atau tidak ada key, pakai []
                        if 'daftarHarga' in entry:
                            for harga_entry in (entry.get('daftarHarga') or []):
                                df_row = pd.json_normalize(harga_entry)
                                df_row['kota']       = kota['nama_kab_kota']
                                df_row['variant_id'] = entry['variant_id']
                                df_row['variant']    = entry['variant']
                                df_row['period']     = f"{start_date} s/d {end_date}"
                                all_data.append(df_row)
                else:
                    print(f"  ⚠ Data kosong untuk {kota['nama_kab_kota']} periode {start_date}")
            else:
                print(f"  ✗ HTTP {response.status_code} untuk {kota['nama_kab_kota']} periode {start_date}")

        except requests.exceptions.RequestException as e:
            print(f"  ✗ Error: {e}")

        # Jeda kecil agar tidak membebani server
        time.sleep(0.3)

# ── Simpan hasil ───────────────────────────────────────────────────────────────
if all_data:
    final_df = pd.concat(all_data, ignore_index=True)
    # Hapus duplikat jika ada overlap data
    final_df.drop_duplicates(inplace=True)
    print(f"\nTotal baris data: {len(final_df)}")
    final_df.to_excel("harga_beras_jatim_2024_2026.xlsx", index=False)
    print("✓ File berhasil disimpan: harga_beras_jatim_2024_2026.xlsx")
else:
    print("Tidak ada data yang berhasil diambil.")

Total rentang bulan: 26
Mengambil: Kab. Pacitan | 2024-01-01 s/d 2024-01-31
Mengambil: Kab. Pacitan | 2024-02-01 s/d 2024-02-29
Mengambil: Kab. Pacitan | 2024-03-01 s/d 2024-03-31
Mengambil: Kab. Pacitan | 2024-04-01 s/d 2024-04-30
Mengambil: Kab. Pacitan | 2024-05-01 s/d 2024-05-31
Mengambil: Kab. Pacitan | 2024-06-01 s/d 2024-06-30
Mengambil: Kab. Pacitan | 2024-07-01 s/d 2024-07-31
Mengambil: Kab. Pacitan | 2024-08-01 s/d 2024-08-31
Mengambil: Kab. Pacitan | 2024-09-01 s/d 2024-09-30
Mengambil: Kab. Pacitan | 2024-10-01 s/d 2024-10-31
Mengambil: Kab. Pacitan | 2024-11-01 s/d 2024-11-30
Mengambil: Kab. Pacitan | 2024-12-01 s/d 2024-12-31
Mengambil: Kab. Pacitan | 2025-01-01 s/d 2025-01-31
Mengambil: Kab. Pacitan | 2025-02-01 s/d 2025-02-28
Mengambil: Kab. Pacitan | 2025-03-01 s/d 2025-03-31
Mengambil: Kab. Pacitan | 2025-04-01 s/d 2025-04-30
Mengambil: Kab. Pacitan | 2025-05-01 s/d 2025-05-31
Mengambil: Kab. Pacitan | 2025-06-01 s/d 2025-06-30
Mengambil: Kab. Pacitan | 2025-07-01 s/d

In [2]:
import requests
import pandas as pd
from datetime import date
from dateutil.relativedelta import relativedelta
import time

# ===============================
# URL API
# ===============================
url = "https://api-sp2kp.kemendag.go.id/report/api/average-price/export-area-daily-json"

headers = {
    "User-Agent": "Mozilla/5.0",
    "Origin": "https://sp2kp.kemendag.go.id",
    "Referer": "https://sp2kp.kemendag.go.id/",
    "Accept": "application/json"
}

# ===============================
# Generate daftar rentang bulan
# ===============================
def generate_monthly_ranges(start: date, end: date):
    ranges = []
    current = start.replace(day=1)

    while current <= end:
        month_start = current
        month_end = (current + relativedelta(months=1)) - relativedelta(days=1)
        month_end = min(month_end, end)

        ranges.append((
            month_start.strftime("%Y-%m-%d"),
            month_end.strftime("%Y-%m-%d")
        ))

        current += relativedelta(months=1)

    return ranges


# ===============================
# Tentukan periode
# ===============================
START_DATE = date(2024, 1, 1)
END_DATE   = date(2026, 2, 18)

monthly_ranges = generate_monthly_ranges(START_DATE, END_DATE)

print(f"Total rentang bulan: {len(monthly_ranges)}")


# ===============================
# Loop ambil data Provinsi Jatim
# ===============================
all_data = []

for start_date, end_date in monthly_ranges:

    print(f"Mengambil Jawa Timur | {start_date} s/d {end_date}")

    payload = {
        "start_date": start_date,
        "end_date": end_date,
        "level": 1,                # ← LEVEL PROVINSI
        "variant_ids": "52,51",
        "kode_provinsi": 35,       # ← 35 = Jawa Timur
        "kode_kab_kota": "",       # ← Kosong
        "pasar_id": "",
        "skip_sat_sun": "true",
        "tipe_komoditas": "1"
    }

    try:
        response = requests.post(url, headers=headers, data=payload, timeout=30)

        if response.status_code == 200:
            data = response.json()

            for entry in (data.get("data") or []):
                for harga_entry in (entry.get("daftarHarga") or []):

                    df_row = pd.json_normalize(harga_entry)

                    df_row["variant_id"] = entry["variant_id"]
                    df_row["variant"] = entry["variant"]
                    df_row["provinsi"] = "Jawa Timur"
                    df_row["period"] = f"{start_date} s/d {end_date}"

                    all_data.append(df_row)

        else:
            print(f"  ✗ HTTP {response.status_code}")

    except requests.exceptions.RequestException as e:
        print(f"  ✗ Error: {e}")

    time.sleep(0.3)


# ===============================
# Gabungkan & Simpan
# ===============================
if all_data:

    final_df = pd.concat(all_data, ignore_index=True)

    # Hapus duplikat jika ada overlap
    final_df.drop_duplicates(inplace=True)

    print(f"\nTotal baris data: {len(final_df)}")

    final_df.to_excel("sp2kp_jatim_2024_2026.xlsx", index=False)

    print("✓ File berhasil disimpan: harga_beras_jatim_provinsi_2024_2026.xlsx")

else:
    print("Tidak ada data yang berhasil diambil.")

Total rentang bulan: 26
Mengambil Jawa Timur | 2024-01-01 s/d 2024-01-31
Mengambil Jawa Timur | 2024-02-01 s/d 2024-02-29
Mengambil Jawa Timur | 2024-03-01 s/d 2024-03-31
Mengambil Jawa Timur | 2024-04-01 s/d 2024-04-30
Mengambil Jawa Timur | 2024-05-01 s/d 2024-05-31
Mengambil Jawa Timur | 2024-06-01 s/d 2024-06-30
Mengambil Jawa Timur | 2024-07-01 s/d 2024-07-31
Mengambil Jawa Timur | 2024-08-01 s/d 2024-08-31
Mengambil Jawa Timur | 2024-09-01 s/d 2024-09-30
Mengambil Jawa Timur | 2024-10-01 s/d 2024-10-31
Mengambil Jawa Timur | 2024-11-01 s/d 2024-11-30
Mengambil Jawa Timur | 2024-12-01 s/d 2024-12-31
Mengambil Jawa Timur | 2025-01-01 s/d 2025-01-31
Mengambil Jawa Timur | 2025-02-01 s/d 2025-02-28
Mengambil Jawa Timur | 2025-03-01 s/d 2025-03-31
Mengambil Jawa Timur | 2025-04-01 s/d 2025-04-30
Mengambil Jawa Timur | 2025-05-01 s/d 2025-05-31
Mengambil Jawa Timur | 2025-06-01 s/d 2025-06-30
Mengambil Jawa Timur | 2025-07-01 s/d 2025-07-31
Mengambil Jawa Timur | 2025-08-01 s/d 2025-08